# Extragere date TheSportsDB — API gratuit

Notebook-ul folosește API-ul V1 cu cheia publică gratuită `123`. Extrage evenimentele (meciurile) pentru o ligă și un sezon, apoi salvează JSON-ul brut și un CSV. API-ul gratuit nu garantează statistici individuale complete pentru fiecare meci.

## 1. Importuri și configurare

Exemplul folosește Premier League (`4328`) și sezonul `2023-2024`. Schimbă valorile dacă vrei altă ligă sau alt sezon.

In [ ]:
from pathlib import Path
import json
import time

import pandas as pd
import requests
from requests.exceptions import HTTPError

API_BASE = "https://www.thesportsdb.com/api/v1/json"
API_KEY = "123"  # cheie gratuită publicată de TheSportsDB
LEAGUE_ID = 4328
SEASON = "2023-2024"
OUTPUT_DIR = Path("data/raw/thesportsdb")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

session = requests.Session()
session.headers.update({"User-Agent": "IAPD-football-data-project/1.0"})


## 2. Funcție pentru cereri API

API-ul gratuit are limită de 30 de cereri pe minut. Notebook-ul face doar câteva cereri pentru catalog și sezon și așteaptă dacă primește HTTP 429.

In [ ]:
def api_get(endpoint: str, params: dict | None = None, max_retries: int = 2) -> dict:
    url = f"{API_BASE}/{API_KEY}/{endpoint}"
    for attempt in range(max_retries + 1):
        response = session.get(url, params=params, timeout=45)
        if response.status_code == 429 and attempt < max_retries:
            wait_seconds = int(response.headers.get("Retry-After", 60))
            print(f"Limită API atinsă; reîncerc peste {wait_seconds} secunde.")
            time.sleep(wait_seconds)
            continue
        try:
            response.raise_for_status()
        except HTTPError as exc:
            raise RuntimeError(f"TheSportsDB HTTP {response.status_code}: {response.text[:400]}") from exc
        try:
            return response.json()
        except ValueError as exc:
            raise RuntimeError("Răspunsul API nu este JSON valid.") from exc
    raise RuntimeError("Nu s-a primit un răspuns valid după reîncercări.")


## 3. Verificarea ligii și a sezoanelor

Confirmă ID-ul ligii și verifică dacă sezonul selectat este listat de API.

In [ ]:
league_payload = api_get("lookupleague.php", {"id": LEAGUE_ID})
league_rows = league_payload.get("leagues") or []
if not league_rows:
    raise ValueError(f"Nu am găsit liga cu ID-ul {LEAGUE_ID}.")
league = league_rows[0]
print("Ligă:", league.get("strLeague"), "| sport:", league.get("strSport"))

seasons_payload = api_get("search_all_seasons.php", {"id": LEAGUE_ID})
seasons = pd.json_normalize(seasons_payload.get("seasons") or [])
display(seasons)


## 4. Extragerea evenimentelor pentru sezon

`eventsseason.php` întoarce evenimentele pentru ID-ul ligii și sezon. Răspunsul brut se păstrează pentru trasabilitate.

In [ ]:
events_payload = api_get("eventsseason.php", {"id": LEAGUE_ID, "s": SEASON})
events = events_payload.get("events") or []

raw_file = OUTPUT_DIR / f"league_{LEAGUE_ID}_{SEASON}_raw.json"
with raw_file.open("w", encoding="utf-8") as f:
    json.dump(events_payload, f, ensure_ascii=False, indent=2)

if not events:
    print("API-ul nu a întors evenimente. Verifică ID-ul ligii, formatul sezonului și acoperirea gratuită.")

events_df = pd.json_normalize(events, sep=".")
csv_file = OUTPUT_DIR / f"league_{LEAGUE_ID}_{SEASON}_events.csv"
events_df.to_csv(csv_file, index=False, encoding="utf-8-sig")
print(f"Evenimente extrase: {len(events_df):,}")
print("CSV:", csv_file.resolve())
display(events_df.head())


## 5. Verificarea câmpurilor disponibile

Inspectează câmpurile returnate înainte să le folosești în analiză. Acoperirea diferă între evenimente și ligi.

In [ ]:
print("Dimensiune:", events_df.shape)
display(events_df.dtypes.to_frame("tip"))
display(events_df.isna().sum().sort_values(ascending=False).head(20).to_frame("valori_lipsa"))


## Limitele planului gratuit

TheSportsDB documentează cheia gratuită `123`, maximum 30 de cereri pe minut și limite diferite pe endpoint. Acest notebook preia programul și detaliile generale ale meciurilor; nu presupune că are minute, șuturi, pase decisive ori cartonașe la nivel de jucător. Lookup-urile de line-up și statistici per eveniment au limite mici pe planul gratuit, așa că nu sunt apelate automat pentru fiecare meci. Vezi [documentația API gratuită](https://api.thesportsdb.com/documentation).